In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import importlib


In [3]:
import copy
import sys
import os

sys.path.append(os.path.abspath("../"))
from campaign_diagram import *

## IN this notebook
# guard_position, dual mode, etc....


A campaign diagram consists of a sequence of *kernels*.
One way to define kernels is to create a `Kernel` object.
Each `Kernel`should have (at minimum):
- a name
- a start time
- the duration
- the compute utilization
- the bandwidth utilization

Let us define three Kernels (Einsums). Einsum A starts at time stamp 0, and ends at time stamp 3. Einsum B starts where Einsum A ended, and continues for 10 more time units (time stamp 13). Einsum C starts where Einsum B ended and continues for 2 more time units (time stamp 15).
<mark> TODO: Change Kernel to something else. Phase? </mark> 
<mark> TODO: try our color picker using colorbrew? Make this an option. </mark>
<mark> TODO: more control over the colors? </mark> 
<mark> TODO: guard_bands too dark, make it a lighter shade. </mark>
Feedback list:
- make the gray area NOT transparent
- make the size of the "above" or "below" the same as "centered" (MP)
  - actually, just cut below and middle :) -> sanity check with JSE
    - because negative utilization has caused high-bit level of confusion ("why is util negative?"), without it it's clear what's going on

In [4]:
kernel1a = Kernel(name='EinsumA',
                  start=0,
                  duration=3,
                  compute_util=0.7,
                  bw_util=0.25)

kernel1b = Kernel(name='EinsumB',
                  start=kernel1a.end,
                  duration=10,
                  compute_util=0.2,
                  bw_util=0.9)

kernel1c = Kernel(name='EinsumC',
                  start=kernel1b.end,
                  duration=2,
                  compute_util=0.6,
                  bw_util=0.4)

Create a `Cascade` (sequence of kernels) by specifying the cascade name and kernels in the cascade.

In [5]:
# Create the plot with a list of kernel instances
cascade1 = Cascade(name="Sample Cascade",
                   kernels=[kernel1a, kernel1b, kernel1c])
cascade1.pretty_print()



Cascade: Sample Cascade
Kernel(name=EinsumA, start=0.00, duration=3.00, throttled_duration=0.90, compute_util=0.70, bw_util=0.25,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=EinsumB, start=3.00, duration=10.00, throttled_duration=1.00, compute_util=0.20, bw_util=0.90,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=EinsumC, start=13.00, duration=2.00, throttled_duration=0.80, compute_util=0.60, bw_util=0.40,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0



To create a basic Campaign Diagram, call `CampaignDiagram` on the cascade.  
Then call `draw()` to display the diagram, and `interactive()` to view the figure interactively.

In [6]:
campaign_diagram1 = CampaignDiagram(cascade1)
campaign_diagram1.draw().interactive()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

In the figure above, each color (blue, orange, green) is an Einsum block.
On the X-axis is time, on the Y-axis, Compute Utilization.  
For a given Einsum, the center, solid line indicates the compute utilization for that block.
The shaded area around the line indicates the memory bandwidth utilization (how much does the shaded, colored area "fill" the memory pipe?). The memory "pipe" is the light grey area, surrounded by two "guard bands" (dark grey lines).

One can also change the visualization to have the memory pipe go in one direction:

In [7]:
campaign_diagram1 = CampaignDiagram(cascade1)
viz = campaign_diagram1.draw(
    guard_position="above",     # or "below", "centered"
    #dual_mode=True,  # dual_mode vs colocated mode. 
    #include_boundaries=True, # draws a vertical line to separate out fusion groups
    return_df=False #True, returns a pandas dataframe of the data used to plot the campaign diagram
)
viz.interactive()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

In [8]:
campaign_diagram1 = CampaignDiagram(cascade1)
viz = campaign_diagram1.draw(
    guard_position="below",     # or "above", "centered"
    #dual_mode=True,  # dual_mode vs colocated mode. 
    #include_boundaries=True, # draws a vertical line to separate out fusion groups
    return_df=False #True, returns a pandas dataframe of the data used to plot the campaign diagram
)
viz.interactive()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

The `draw` method can also return a Pandas dataframe of the data used to create the campaign diagram.

In [9]:
campaign_diagram1 = CampaignDiagram(cascade1)
viz, df = campaign_diagram1.draw(
    guard_position="below",     # or "above", "centered"
    #dual_mode=True,  # dual_mode vs colocated mode. 
    #include_boundaries=True, # draws a vertical line to separate out fusion groups
    return_df=True #True, returns a pandas dataframe of the data used to plot the campaign diagram
)
viz.interactive().show()
df

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

,Einsum,fusion_group,fusion_type,Starting_Time,Time_Stamp,End_Time,cb_End_Time,mb_End_Time,full_End_Time,runtime,...,compute_color,bw_color,throttled,throttled_duration,mem_throttled_duration,comp_throttled_duration,comp_perc,bw_perc,compute_type,memory_type
0,EinsumA,None,NONE,0.0,0.0,2.1,2.1,0.75,3.0,3,...,#332288,#9990C3,True,0.9,2.25,0.9,1.0,1.0,Compute Pool,Memory Pool
1,EinsumB,None,NONE,3.0,3.0,12.0,5.0,12.00,13.0,10,...,#117733,#88BB99,True,1.0,1.00,8.0,1.0,1.0,Compute Pool,Memory Pool
2,EinsumC,None,NONE,13.0,13.0,14.2,14.2,13.80,15.0,2,...,#882255,#C390AA,True,0.8,1.20,0.8,1.0,1.0,Compute Pool,Memory Pool


The above visualizations are done using the "colocated" view of the campaign diagrams.  
    - <mark> TODO: define colocated -- both mem and comp are on a single dimension on the page </mark>
We have a second view, called the "dual mode" view.
In the dual mode, Compute Utilization is on the positive Y-axis, and Memory Bandwidth Utilization is on the Negative Y-axis.

- <mark> TODO: add a thick line at 0 </mark>

In [10]:
campaign_diagram1 = CampaignDiagram(cascade1)
viz, df = campaign_diagram1.draw(
    guard_position="below",     # or "above", "centered" -- irrelevant in dual _mode
    dual_mode=True,  # dual_mode vs colocated mode. 
    #include_boundaries=True, # draws a vertical line to separate out fusion groups
    return_df=True #True, returns a pandas dataframe of the data used to plot the campaign diagram
)
viz.interactive().show()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

<mark> TODO: paragraph on 'what is this actually showing me?' 3 phases, width is latency, compute <>, memory <>, conclusion <> </mark>

In [11]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import importlib


In [13]:
import copy
import sys
import os

sys.path.append(os.path.abspath("../"))
from campaign_diagram import *

## IN this notebook
# guard_position, dual mode, etc....


A campaign diagram consists of a sequence of *kernels*.
One way to define kernels is to create a `Kernel` object.
Each `Kernel`should have (at minimum):
- a name
- a start time
- the duration
- the compute utilization
- the bandwidth utilization

Let us define three Kernels (Einsums). Einsum A starts at time stamp 0, and ends at time stamp 3. Einsum B starts where Einsum A ended, and continues for 10 more time units (time stamp 13). Einsum C starts where Einsum B ended and continues for 2 more time units (time stamp 15).
<mark> TODO: Change Kernel to something else. Phase? </mark> 
<mark> TODO: try our color picker using colorbrew? Make this an option. </mark>
<mark> TODO: more control over the colors? </mark> 
<mark> TODO: guard_bands too dark, make it a lighter shade. </mark>
Feedback list:
- make the gray area NOT transparent
- make the size of the "above" or "below" the same as "centered" (MP)
  - actually, just cut below and middle :) -> sanity check with JSE
    - because negative utilization has caused high-bit level of confusion ("why is util negative?"), without it it's clear what's going on

In [14]:
kernel1a = Kernel(name='EinsumA',
                  start=0,
                  duration=3,
                  compute_util=0.7,
                  bw_util=0.25)

kernel1b = Kernel(name='EinsumB',
                  start=kernel1a.end,
                  duration=10,
                  compute_util=0.2,
                  bw_util=0.9)

kernel1c = Kernel(name='EinsumC',
                  start=kernel1b.end,
                  duration=2,
                  compute_util=0.6,
                  bw_util=0.4)

Create a `Cascade` (sequence of kernels) by specifying the cascade name and kernels in the cascade.

In [15]:
# Create the plot with a list of kernel instances
cascade1 = Cascade(name="Sample Cascade",
                   kernels=[kernel1a, kernel1b, kernel1c])
cascade1.pretty_print()



Cascade: Sample Cascade
Kernel(name=EinsumA, start=0.00, duration=3.00, throttled_duration=0.90, compute_util=0.70, bw_util=0.25,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=EinsumB, start=3.00, duration=10.00, throttled_duration=1.00, compute_util=0.20, bw_util=0.90,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=EinsumC, start=13.00, duration=2.00, throttled_duration=0.80, compute_util=0.60, bw_util=0.40,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0



To create a basic Campaign Diagram, call `CampaignDiagram` on the cascade.  
Then call `draw()` to display the diagram, and `interactive()` to view the figure interactively.

In [16]:
campaign_diagram1 = CampaignDiagram(cascade1)
campaign_diagram1.draw().interactive()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

In the figure above, each color (blue, orange, green) is an Einsum block.
On the X-axis is time, on the Y-axis, Compute Utilization.  
For a given Einsum, the center, solid line indicates the compute utilization for that block.
The shaded area around the line indicates the memory bandwidth utilization (how much does the shaded, colored area "fill" the memory pipe?). The memory "pipe" is the light grey area, surrounded by two "guard bands" (dark grey lines).

One can also change the visualization to have the memory pipe go in one direction:

In [17]:
campaign_diagram1 = CampaignDiagram(cascade1)
viz = campaign_diagram1.draw(
    guard_position="above",     # or "below", "centered"
    #dual_mode=True,  # dual_mode vs colocated mode. 
    #include_boundaries=True, # draws a vertical line to separate out fusion groups
    return_df=False #True, returns a pandas dataframe of the data used to plot the campaign diagram
)
viz.interactive()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

In [18]:
campaign_diagram1 = CampaignDiagram(cascade1)
viz = campaign_diagram1.draw(
    guard_position="below",     # or "above", "centered"
    #dual_mode=True,  # dual_mode vs colocated mode. 
    #include_boundaries=True, # draws a vertical line to separate out fusion groups
    return_df=False #True, returns a pandas dataframe of the data used to plot the campaign diagram
)
viz.interactive()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

The `draw` method can also return a Pandas dataframe of the data used to create the campaign diagram.

In [19]:
campaign_diagram1 = CampaignDiagram(cascade1)
viz, df = campaign_diagram1.draw(
    guard_position="below",     # or "above", "centered"
    #dual_mode=True,  # dual_mode vs colocated mode. 
    #include_boundaries=True, # draws a vertical line to separate out fusion groups
    return_df=True #True, returns a pandas dataframe of the data used to plot the campaign diagram
)
viz.interactive().show()
df

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

,Einsum,fusion_group,fusion_type,Starting_Time,Time_Stamp,End_Time,cb_End_Time,mb_End_Time,full_End_Time,runtime,...,compute_color,bw_color,throttled,throttled_duration,mem_throttled_duration,comp_throttled_duration,comp_perc,bw_perc,compute_type,memory_type
0,EinsumA,None,NONE,0.0,0.0,2.1,2.1,0.75,3.0,3,...,#332288,#9990C3,True,0.9,2.25,0.9,1.0,1.0,Compute Pool,Memory Pool
1,EinsumB,None,NONE,3.0,3.0,12.0,5.0,12.00,13.0,10,...,#117733,#88BB99,True,1.0,1.00,8.0,1.0,1.0,Compute Pool,Memory Pool
2,EinsumC,None,NONE,13.0,13.0,14.2,14.2,13.80,15.0,2,...,#882255,#C390AA,True,0.8,1.20,0.8,1.0,1.0,Compute Pool,Memory Pool


The above visualizations are done using the "colocated" view of the campaign diagrams.  
    - <mark> TODO: define colocated -- both mem and comp are on a single dimension on the page </mark>
We have a second view, called the "dual mode" view.
In the dual mode, Compute Utilization is on the positive Y-axis, and Memory Bandwidth Utilization is on the Negative Y-axis.

- <mark> TODO: add a thick line at 0 </mark>

In [20]:
campaign_diagram1 = CampaignDiagram(cascade1)
viz, df = campaign_diagram1.draw(
    guard_position="below",     # or "above", "centered" -- irrelevant in dual _mode
    dual_mode=True,  # dual_mode vs colocated mode. 
    #include_boundaries=True, # draws a vertical line to separate out fusion groups
    return_df=True #True, returns a pandas dataframe of the data used to plot the campaign diagram
)
viz.interactive().show()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

<mark> TODO: paragraph on 'what is this actually showing me?' 3 phases, width is latency, compute <>, memory <>, conclusion <> </mark>